# **[Microsoft Fabric: Implementing Security in Fabric Warehouses](https://accenture.percipio.com/courses/db811b8a-9035-4037-9e33-9f1231af76e0)**

## **1. Course Overview**
### Explanation
This course focuses on securing Fabric Warehouses using multiple security layers. It covers workspace roles, object-level permissions, Column-Level Security (CLS), Row-Level Security (RLS), Dynamic Data Masking (DDM), Views, Functions, Stored Procedures, and Dynamic SQL. The course emphasizes the Principle of Least Privilege, one of the most important DP-600 concepts.
### Summary
* Workspace Roles
* GRANT, DENY, REVOKE
* Least Privilege Principle
* Column-Level Security (CLS)
* Row-Level Security (RLS)
* Dynamic Data Masking (DDM)
* Views
* Functions
* Stored Procedures
* Dynamic SQL Queries
* Advanced Warehouse Security

## **2. Security Features in Warehouses**
### Explanation
Fabric provides multiple layers of security:
### Workspace Roles
Control access at workspace level.
### Object-Level Security (OLS)
Controls access to:
* Tables
* Views
* Stored Procedures
### Column-Level Security (CLS)
Controls access to specific columns.
### Row-Level Security (RLS)
Controls access to specific rows.
### Dynamic Data Masking (DDM)
Masks sensitive values without modifying data.
### Auditing
Uses:
* Microsoft Purview
* PowerShell
* Audit Logs
### Summary
Fabric security is implemented through workspace access controls, object permissions, row/column filtering, masking, and auditing.

## **3. Assigning Workspace Roles**
### Explanation
Workspace roles apply to all items within a workspace.
| Role | Permissions |
| --- | --- |
| Admin | Full control |
| Member | Create & manage content |
| Contributor | Create & modify content |
| Viewer | Read-only access |
### Key Findings
#### Viewer
* Read data
* Cannot modify content
#### Contributor
* Insert
* Update
* Delete
* Modify warehouse data
#### Admin
* Manage workspace
* Manage users
* Full warehouse control
### Summary
Workspace roles provide the first layer of access control in Fabric.

## **4. Using `GRANT`, `DENY`, and `REVOKE`**
### Explanation
Used for object-level permissions.
### GRANT
Allows access:
```sql
GRANT INSERT ON SupermarketSales
```
### DENY
Explicitly blocks access:
```sql
DENY DELETE ON SupermarketSales
```
### REVOKE
Removes a previous GRANT or DENY.
```sql
REVOKE UPDATE
```
Returns user access to workspace-level permissions.
### Summary
* `GRANT` = Allow
* `DENY` = Block
* `REVOKE` = Remove Explicit Rule

## **5. Principle of Least Privilege**
### Explanation
Users should receive only the permissions necessary to perform their job.
### Example:
Application should:
* Execute Stored Procedure

Application should NOT:
* Query underlying table directly
### Implementation
```sql
DENY SELECT ON SupermarketSales

GRANT EXECUTE
ON StoredProcedure
```
### DP-600 Exam Tip
> Least Privilege = Major security principle.

### Summary
Grant access to business functions, not underlying data.

## **6. Configuring Column-Level Security (CLS)**
### Explanation
CLS protects sensitive columns.
### Example:
Deny access to:
* Salary
* Payment Details
* Customer Type

While allowing access to:
* Product
* Date
* Location
### Method 1
Deny specific columns:
```sql
DENY SELECT
ON SupermarketSales
(InvoiceID, CustomerType, Payment)
```
### Method 2
Deny entire table then allow selected columns.
```sql
DENY SELECT ON SupermarketSales

GRANT SELECT
ON specific columns
```
### Summary
CLS is used when users need access to a table but not all columns.

## **7. Configuring Row-Level Security (RLS)**
### Explanation
RLS restricts rows visible to specific users.
### Example:
User:
* Contact

Can see:
* Health & Beauty products

Cannot see:
* Electronics
* Fashion
* Food

### Step 1: Create Predicate Function
```sql
CREATE FUNCTION SalesPredicate()
```
Determines access.

Returns:
* 1 → Allow
* No rows → Deny
### Step 2: Create Security Policy
```sql
CREATE SECURITY POLICY
```
Applies predicate to table.
### Summary
RLS filters rows transparently based on user identity.

## **8. Operations Supported Under RLS**
### Explanation
Users can operate only on visible rows.

Allowed:
* `SELECT`
* `UPDATE`
* `DELETE`

Only for rows returned by RLS.
### Example
User can:
```sql
UPDATE
Health & Beauty
```
User cannot:
```sql
UPDATE
Fashion Accessories
```
because those rows are invisible.
### Summary
RLS affects all queries and modifications by filtering visible rows.

## **9. Altering Row-Level Security**
### Explanation
To modify RLS:
#### Step 1
Drop existing policy
```sql
DROP SECURITY POLICY
```
#### Step 2
Alter predicate
```sql
ALTER FUNCTION
```
#### Step 3
Recreate policy
```sql
CREATE SECURITY POLICY
```
### Enable / Disable
```sql
STATE = ON

STATE = OFF
```
### Summary
RLS policies are maintained through Predicate Functions + Security Policies.

## **10. Applying Dynamic Data Masking (DDM)**
### Explanation
DDM hides sensitive values while leaving actual data unchanged.
#### Types of masks:
##### Default()
Example:
```sql
Salary → 0
```
##### Email()
```sql
john@example.com
```
becomes
```sql
jXXX@XXXX.com
```
##### Random()
Returns random value.

Example:
```sql
random(1,5)
```
##### Partial()
Custom masking pattern.

Example:
```sql
123-456-7890
```
becomes
```sql
123XXX-XXXX
```
### Summary
DDM protects sensitive data without changing stored values.

## **11. Viewing Unmasked Data**
### Explanation
Unmasked access can be granted:
```sql
GRANT UNMASK
```
### Important Rule
These workspace roles always see unmasked data:
* Admin
* Member
* Contributor

Masked data primarily affects:
* Viewer role
### Summary
`UNMASK` permission overrides Dynamic Data Masking restrictions.

## **12. Advanced Query Constructs**
### Explanation
Four major constructs:
#### Views
Virtual tables.
#### Functions
Reusable read-only logic.
#### Stored Procedures
Reusable read-write logic.
#### Dynamic SQL
Query generated at runtime.
### Summary
Each construct serves different security and reuse purposes.

## **13. Creating and Querying Views**
### Explanation
Views are virtual tables.
#### Example:
```sql
CREATE VIEW YangonSales
```
based on:
```sql
SELECT *
FROM SupermarketSales
WHERE City='Yangon'
```
### Benefits
* Simplifies queries
* Improves security
* Creates business-specific datasets
### Limitation
Views do not store data.
### Summary
Views expose filtered business-friendly datasets without duplicating data.

## **14. Creating Functions**
### Explanation
Fabric supports: \
✅ Table-Valued Functions (TVFs)

Fabric does NOT support: \
❌ Scalar Functions

### Example
```sql
CREATE FUNCTION
```
returns:
|Product | TotalSales |
| --- | --- |

instead of a single value.

### Key Limitation
Functions:
* Read-only
* Cannot UPDATE
* Cannot INSERT
* Cannot DELETE
### Summary
Functions provide reusable query logic but cannot modify data.

## **15. Creating Stored Procedures**
### Explanation
Stored Procedures:
* Precompiled
* Accept parameters
* Can Perform DML
```sql
EXEC StoredProcedure
```
### Capabilities
✅ SELECT \
✅ INSERT \
✅ UPDATE \
✅ DELETE \
✅ Transactions

### Security Benefit
Users can:
* Execute procedure

without
* Direct table access
### Summary
Stored Procedures are central to implementing least-privilege designs.

## **16. Dynamic SQL Queries**
### Explanation
Dynamic SQL generates query logic at runtime.
### Preferred method:
```sql
sp_executesql
```
because:
* Parameterized
* Safer
* Prevents SQL Injection
### Not Recommended
```sql
EXEC(@sql)
```
because:
* Vulnerable to SQL Injection
### Summary
Always prefer `sp_executesql` when creating parameterized dynamic queries.

## **17. Course Summary**
### Key Topics Covered
✅ Workspace Roles \
✅ Object-Level Security (OLS) \
✅ Column-Level Security (CLS) \
✅ Row-Level Security (RLS) \
✅ Dynamic Data Masking (DDM) \
✅ GRANT / DENY / REVOKE \
✅ Principle of Least Privilege \
✅ UNMASK Permission \
✅ Views \
✅ Functions \
✅ Stored Procedures \
✅ Dynamic Queries \
✅ Security Policies & Predicates
### DP-600 High-Priority Revision Topics
| Topic | Priority |
| --- | --- |
| Workspace Roles | ⭐⭐⭐⭐⭐ |
| GRANT / DENY / REVOKE | ⭐⭐⭐⭐⭐ |
| Least Privilege | ⭐⭐⭐⭐⭐ |
| Row-Level Security | ⭐⭐⭐⭐⭐ |
| Column-Level Security | ⭐⭐⭐⭐⭐ |
| Dynamic Data Masking | ⭐⭐⭐⭐⭐ |
| UNMASK Permission | ⭐⭐⭐⭐ |
| Views | ⭐⭐⭐⭐ |
| Table-Valued Functions | ⭐⭐⭐⭐ |
| Stored Procedures | ⭐⭐⭐⭐⭐ |
| Dynamic SQL | ⭐⭐⭐⭐ |
### Quick Revision Notes
* Admin > Member > Contributor > Viewer
* `GRANT` = Allow
* `DENY` = Block
* `REVOKE` = Remove Explicit Permission
* CLS = Restrict Columns
* RLS = Restrict Rows
* DDM = Mask Values
* UNMASK = Reveal Values
* Views = Virtual Tables
* Functions = Read Only
* Stored Procedures = Read + Write
* Use `sp_executesql` for Dynamic SQL
* Principle of Least Privilege = DP-600 Favorite Topic

## **Quiz**

### In Microsoft Fabric, how can you enforce column-level security to ensure that users can only access specific columns in a data warehouse table? 

A) Create a stored procedure to mask sensitive data in the specific columns \
B) Apply a `WHERE` clause to filter rows which have values in those columns \
C) Use `GRANT SELECT ON` the table without specifying columns \
**D) Use `GRANT SELECT ON` specific columns to restrict access to only those columns**

### What does this code accomplish, and how does it relate to the principle of least privilege?
```sql
DENY SELECT ON SupermarketSales TO [contact@loonycorn.com]; 
GRANT EXECUTE ON GetSalesByBranchAndDate TO [contact@loonycorn.com]; 
```
A) It grants full access to the `SupermarketSales` table and the `GetSalesByBranchAndDate` stored procedure, ensuring no restrictions on data access violating to the principle of least privilege \
B) It provides full access to both the table and the stored procedure, violating the principle of least privilege \
C) It denies the user from executing any stored procedures but allows them to select from the `SupermarketSales` table adhering to the principle of least privilege \
**D) It denies direct `SELECT` access to the `SupermarketSales` table but allows the user to run the stored procedure `GetSalesByBranchAndDate` adhering to the principle of least privilege**

### Which function or stored procedure is used to execute dynamic SQL statements in Microsoft Fabric warehouses in a parameterized manner that is safe from SQL injection attacks?

A) `RUNSQL` \
**B) `sp_executesql`** \
C) `EXEC_DYNAMIC` \
D) `EXEC`

### How would you alter an existing security predicate that you previously configured on a table to enforce row-level security in Microsoft Fabric?

A) Use the `UPDATE PREDICATE` command to change the row-level security policy \
**B) Drop the existing security policy and create a new one with the updated predicate** \
C) Use the `ALTER SECURITY PREDICATE` statement to modify the predicate directly \
D) Modify the row-level security by editing the table schema

### Which type of function is supported in Microsoft Fabric warehouses?

A) Neither table-valued nor scalar-valued functions are supported \
**B) Only table-valued functions that return a set of rows** \
C) Both scalar-valued and table-valued functions \
D) Only scalar-valued functions that return a single value

### Please match the sources which define the workspace role with the permissions they are allowed:
#### Can view content
A) Contributor \
**B) Viewer**
#### Can edit content
**A) Contributor** \
B) Viewer
#### Can run read queries on warehouses
A) Contributor \
**B) Viewer**
#### Can run insert queries on warehouses
**A) Contributor** \
B) Viewer

### Which of the following are true characteristics of views in Microsoft Fabric warehouse tables?

A) Views require manual updates whenever the underlying table data is modified \
**B) Views can be used to simplify complex queries by encapsulating them into a single object** \
**C) Views do not store data physically; they represent a saved query that retrieves data from the underlying tables** \
D) Views automatically update the underlying data in the table when a change is made to the view

### What is a key characteristic of stored procedures in Microsoft Fabric warehouses?

A) They are limited to only selecting data from a single table \
B) They cannot accept input parameters in Microsoft Fabric \
**C) Stored procedures can perform data manipulation operations on tables** \
D) Stored procedures automatically run every time new data is inserted into a table

### Which of the following statements is true about data masking in Microsoft Fabric workspaces?

A) Masking applies equally to all roles, including Contributors \
B) A Viewer on a workspace will always see unmasked data \
**C) A Contributor on a workspace will always see unmasked data** \
D) Data masking is only applied to external users, not internal workspace roles

### In Microsoft Fabric, which type of query allows you to construct SQL statements programmatically to adjust the query based on variable inputs or conditions at runtime?

A) Stored procedures \
B) Static SQL queries \
**C) Dynamic SQL queries** \
D) Parameterized queries

### In Microsoft Fabric warehouses, which security feature helps protect sensitive data by partially hiding it from unauthorized users while allowing access to authorized users?

**A) Dynamic data masking** \
B) Multi-factor authentication \
C) Row-level security \
D) Data encryption at rest

### In Microsoft Fabric warehouses, which scenarios best describe when you would use the MASKED WITH (`FUNCTION = 'partial()'`) function for data masking?

A) To replace an entire sensitive value, such as a Social Security Number, with a random string of characters \
B) To completely hide the value of a sensitive field and replace it with a default value like "XXXX" \
C) To apply a fixed mask to an email address, such as converting "john.doe@example.com" to "jXXX.dXXX@example.com" \
**D) To display only the first and last few characters of a sensitive value, while masking the middle part, such as showing "`J***s`" for the name "James"**

### In Microsoft Fabric, which command would you use to remove previously granted object-level permissions without denying them explicitly?

**A) `REVOKE`** \
B) `DENY` \
C) `UNGRANT` \
D) `REMOVE`

### Which operations are supported for a user who is restricted by row-level security applied on a table in a Microsoft Fabric warehouse?

A) Update all records in the table, regardless of the security policy \
**B) View only the records permitted by the security policy** \
**C) Update only the records permitted by the security policy** \
D) Delete all records in the table, bypassing the security policy

### In Microsoft Fabric, what feature would you use to restrict access to specific records in a table based on the user’s identity or role?

A) Data Masking \
B) Column-Level Security \
C) Table Permissions \
**D) Row-Level Security**